[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/02.%20Parte%202/09.%20Clase%209/09Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=02.%20Parte%202%2F09.%20Clase%209%2F09Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 9: Optimización de portafolios — Monte Carlo vs. Markowitz

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Simular portafolios aleatorios mediante **Monte Carlo** con estimadores robustos.
- Comparar la simulación MC con la **frontera eficiente** de Markowitz (CVXPY/DCP).
- Identificar el portafolio de **máximo Sharpe** y de **mínima varianza** por ambos métodos.
- Entender por qué la optimización convexa supera al muestreo aleatorio (Boyd & Vandenberghe, 2004, §4.4).

---

## Introducción teórica

### Monte Carlo vs. optimización determinista

En la Parte 1 (Clases 4–8) usamos dos enfoques para encontrar portafolios óptimos:

| Enfoque | Método | Ventaja | Limitación |
|---------|--------|---------|------------|
| **Monte Carlo** | Generar $N$ portafolios aleatorios y evaluar cada uno | Explora todo el espacio factible, visualiza la "nube" | Ineficiente: necesita $N \to \infty$ para acercarse al óptimo |
| **Markowitz (QP)** | Resolver min $\mathbf{w}^\top \Sigma \mathbf{w}$ con CVXPY | Encuentra el óptimo **exacto** en tiempo polinomial | Sensible a errores de estimación en $\mu$ y $\Sigma$ |

### Enfoque refinado de esta clase

Combinamos lo mejor de ambos:
1. **MC con estimadores robustos**: usamos Huber (Clase 6) para $\mu$ y Ledoit-Wolf (Clase 5) para $\Sigma$.
2. **Frontera eficiente con CVXPY**: resolvemos el QP paramétrico exacto con los mismos estimadores.
3. **Superposición**: visualizamos la nube MC junto con la frontera eficiente para confirmar que la optimización encuentra el borde superior de la nube.

> La simulación MC es valiosa para **explorar** el espacio de portafolios y visualizar la relación rendimiento-riesgo. La optimización convexa es necesaria para **encontrar** el portafolio óptimo exacto (Boyd & Vandenberghe, 2004, §4.4, §11.1).

## 1. Descarga de datos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.covariance as skcov
import statsmodels.api as sm

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 10)

In [ ]:
tickers = ["AAPL", "AMZN", "MSFT", "KO"]
start_date = "2025-01-01"
end_date   = "2025-03-27"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()

fig, ax = plt.subplots()
(closes / closes.iloc[0] * 100).plot(ax=ax)
ax.set_title("Precios normalizados (base 100)")
ax.set_ylabel("Índice")
plt.tight_layout()

---
## 2. Rendimientos y estimadores robustos

In [ ]:
daily_returns = np.log(closes / closes.shift(1)).dropna()

# Estimadores robustos
huber = sm.robust.scale.Huber()
mu_huber = np.array([huber(daily_returns[col].values)[0] for col in daily_returns.columns])
Sigma_lw = skcov.LedoitWolf().fit(daily_returns).covariance_

# Comparar con muestrales
mu_sample = daily_returns.mean().values

print("Rendimientos esperados diarios:")
for ticker, m_s, m_h in zip(tickers, mu_sample, mu_huber):
    print(f"  {ticker}: muestral = {m_s:.6f}, Huber = {m_h:.6f}")

In [ ]:
# Covarianza: muestral vs Ledoit-Wolf
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pd.DataFrame(np.cov(daily_returns.T), index=tickers, columns=tickers),
            annot=True, fmt=".6f", cmap="coolwarm", ax=axes[0], square=True)
axes[0].set_title("Covarianza muestral")

sns.heatmap(pd.DataFrame(Sigma_lw, index=tickers, columns=tickers),
            annot=True, fmt=".6f", cmap="coolwarm", ax=axes[1], square=True)
axes[1].set_title("Covarianza Ledoit-Wolf")
plt.tight_layout()

---
## 3. Simulación Monte Carlo de portafolios

### Generación de pesos aleatorios

Generamos $N$ vectores de pesos $\mathbf{w}^{(i)} \sim \text{Dirichlet}(\mathbf{1})$ que satisfacen automáticamente $\sum w_j = 1$ y $w_j \geq 0$. Para cada portafolio calculamos:

$$
\mu_p^{(i)} = 252 \, \hat{\boldsymbol{\mu}}^\top \mathbf{w}^{(i)}, \qquad \sigma_p^{(i)} = \sqrt{252 \, (\mathbf{w}^{(i)})^\top \hat{\Sigma} \, \mathbf{w}^{(i)}}
$$

donde $\hat{\boldsymbol{\mu}}$ y $\hat{\Sigma}$ son los estimadores robustos (Huber y Ledoit-Wolf).

In [ ]:
np.random.seed(42)
n_assets = len(tickers)
n_portfolios = 25000

# Pesos aleatorios (Dirichlet)
weights = np.random.dirichlet(np.ones(n_assets), n_portfolios)

# Rendimiento y riesgo anualizado con estimadores robustos
port_returns = 252 * weights @ mu_huber
port_risks = np.array([np.sqrt(252 * w @ Sigma_lw @ w) for w in weights])
sharpe = port_returns / port_risks

# Portafolios destacados
idx_max_sharpe = sharpe.argmax()
idx_min_vol = port_risks.argmin()

print(f"Monte Carlo — {n_portfolios:,} portafolios simulados")
print(f"  Max Sharpe: S = {sharpe[idx_max_sharpe]:.4f}, μ = {port_returns[idx_max_sharpe]:.4f}, σ = {port_risks[idx_max_sharpe]:.4f}")
print(f"  Min Vol:    σ = {port_risks[idx_min_vol]:.4f}, μ = {port_returns[idx_min_vol]:.4f}")

In [ ]:
# Visualización
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(port_risks, port_returns, c=sharpe, cmap="RdYlBu",
                     s=5, alpha=0.5)
plt.colorbar(scatter, label="Ratio de Sharpe")

# Max Sharpe
ax.scatter(port_risks[idx_max_sharpe], port_returns[idx_max_sharpe],
           marker="*", s=400, color="red", zorder=5, edgecolors="black",
           label=f"Max Sharpe MC (S={sharpe[idx_max_sharpe]:.3f})")
# Min Vol
ax.scatter(port_risks[idx_min_vol], port_returns[idx_min_vol],
           marker="*", s=400, color="green", zorder=5, edgecolors="black",
           label=f"Min Vol MC (σ={port_risks[idx_min_vol]:.3f})")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title(f"Monte Carlo: {n_portfolios:,} portafolios (μ Huber + Σ Ledoit-Wolf)")
ax.legend()
plt.tight_layout()

In [ ]:
# Pesos del portafolio de máximo Sharpe (MC)
mc_sharpe_weights = pd.DataFrame({
    "Activo": tickers,
    "Peso": weights[idx_max_sharpe]
}).set_index("Activo")
mc_sharpe_weights

---
## 4. Frontera eficiente con CVXPY (DCP)

### Comparación con Monte Carlo

Ahora resolvemos el problema de forma **exacta** usando programación cuadrática paramétrica. La frontera eficiente es la curva que envuelve por arriba la nube de portafolios simulados.

In [ ]:
def calc_efficient_frontier(mu_vec, Sigma, n_points=200):
    """Frontera eficiente con CVXPY (DCP)."""
    n = len(mu_vec)
    w = cp.Variable(n)
    target = cp.Parameter()
    risk = cp.quad_form(w, Sigma)
    prob = cp.Problem(cp.Minimize(risk),
                      [mu_vec @ w == target, cp.sum(w) == 1, w >= 0])

    means, stds = [], []
    for r in np.linspace(mu_vec.min(), mu_vec.max(), n_points):
        target.value = r
        prob.solve(solver=cp.ECOS, warm_start=True)
        if prob.status == "optimal":
            means.append(252 * r)
            stds.append(np.sqrt(252 * risk.value))
    return means, stds

# Frontera con estimadores robustos
ef_means, ef_stds = calc_efficient_frontier(mu_huber, Sigma_lw)
print(f"Frontera eficiente: {len(ef_means)} puntos")

In [ ]:
def max_sharpe_cvxpy(mu_vec, Sigma, rf=0.0):
    """Portafolio de máximo Sharpe via transformación DCP."""
    n = len(mu_vec)
    excess = mu_vec - rf
    y = cp.Variable(n)
    kappa = cp.Variable(nonneg=True)
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, Sigma)),
                      [excess @ y == 1, cp.sum(y) == kappa, y >= 0])
    prob.solve(solver=cp.ECOS)
    w_opt = y.value / kappa.value
    ret = 252 * mu_vec @ w_opt
    risk = np.sqrt(252 * w_opt @ Sigma @ w_opt)
    return w_opt, ret, risk, ret / risk

w_cvxpy, ret_cvxpy, risk_cvxpy, sharpe_cvxpy = max_sharpe_cvxpy(mu_huber, Sigma_lw)
print(f"CVXPY Max Sharpe: S = {sharpe_cvxpy:.4f}, μ = {ret_cvxpy:.4f}, σ = {risk_cvxpy:.4f}")

---
## 5. Superposición: Monte Carlo + frontera eficiente

La frontera eficiente (línea azul) envuelve perfectamente la nube de portafolios simulados, confirmando que la optimización convexa encuentra el borde óptimo del espacio factible.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Nube MC
scatter = ax.scatter(port_risks, port_returns, c=sharpe, cmap="RdYlBu",
                     s=5, alpha=0.3)
plt.colorbar(scatter, label="Sharpe (MC)")

# Frontera eficiente
ax.plot(ef_stds, ef_means, "b-", linewidth=2.5, label="Frontera eficiente (CVXPY)")

# Max Sharpe MC
ax.scatter(port_risks[idx_max_sharpe], port_returns[idx_max_sharpe],
           marker="*", s=300, color="red", zorder=5, edgecolors="black",
           label=f"Max Sharpe MC (S={sharpe[idx_max_sharpe]:.3f})")

# Max Sharpe CVXPY
ax.scatter(risk_cvxpy, ret_cvxpy, marker="D", s=150, color="gold", zorder=5,
           edgecolors="black", linewidths=1.5,
           label=f"Max Sharpe CVXPY (S={sharpe_cvxpy:.3f})")

# Activos individuales
for i, ticker in enumerate(tickers):
    ax.scatter(daily_returns[ticker].std() * np.sqrt(252),
               daily_returns[ticker].mean() * 252, s=80, zorder=5)
    ax.annotate(ticker, (daily_returns[ticker].std() * np.sqrt(252),
                         daily_returns[ticker].mean() * 252),
                fontsize=11, ha="left", va="bottom")

ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Monte Carlo vs. Frontera Eficiente (Markowitz/CVXPY)")
ax.legend(loc="upper left")
plt.tight_layout()

---
## 6. Comparación de pesos óptimos

In [ ]:
comparison = pd.DataFrame({
    "Monte Carlo": weights[idx_max_sharpe],
    "CVXPY": w_cvxpy,
    "Diferencia": w_cvxpy - weights[idx_max_sharpe]
}, index=tickers)

print("Pesos del portafolio de máximo Sharpe:")
print(comparison)
print(f"\nSharpe MC:    {sharpe[idx_max_sharpe]:.4f}")
print(f"Sharpe CVXPY: {sharpe_cvxpy:.4f}")
print(f"Mejora:       {(sharpe_cvxpy - sharpe[idx_max_sharpe]) / sharpe[idx_max_sharpe] * 100:+.2f}%")

In [ ]:
comparison[["Monte Carlo", "CVXPY"]].plot(kind="bar", figsize=(10, 5))
plt.title("Pesos óptimos (max Sharpe): Monte Carlo vs. CVXPY")
plt.ylabel("Peso")
plt.xticks(rotation=0)
plt.tight_layout()

---

## Navegación del curso

← **Anterior**: Clase 8: Resumen Parte 1  
→ **Siguiente**: Clase 10: Inclusión de activo libre de riesgo

---

## 7. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press. — §4.4 (QP), §4.7.3 (optimización paramétrica), §11.1 (métodos de punto interior).
- **Glasserman, P.** (2003). *Monte Carlo Methods in Financial Engineering*. Springer.
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.